### LightGBM

In [ ]:
# Define objective function: calculate mean R² score using 5-fold cross-validation
def lgb_cv(n_estimators, max_depth, learning_rate, num_leaves, subsample, colsample_bytree, reg_lambda, reg_alpha):
    # Restrict parameters to reasonable ranges
    n_estimators = int(n_estimators)
    max_depth = int(max_depth)
    num_leaves = int(num_leaves)
    learning_rate = max(min(learning_rate, 1.0), 0.01)  # Limit between 0.01 and 1.0
    subsample = max(min(subsample, 1.0), 0.1)           # Limit between 0.1 and 1.0
    colsample_bytree = max(min(colsample_bytree, 1.0), 0.1)  # Limit between 0.1 and 1.0
    reg_lambda = max(reg_lambda, 0)                     # Minimum is 0
    reg_alpha = max(reg_alpha, 0)                       # Minimum is 0
    
    model = LGBMRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        random_state=42,
        n_jobs=-1,
        verbose=-1  # Disable all LightGBM logs, including warnings
    )
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return cv_score

# Define parameter space
pbounds = {
    'n_estimators': (400, 800),      # Expand range to ensure enough trees
    'max_depth': (5, 15),            # Narrow range to prevent overfitting
    'learning_rate': (0.05, 0.3),    # Raise lower bound to ensure learning rate is not below 0.05
    'num_leaves': (30, 70),          # Narrow range to prevent too many leaves
    'subsample': (0.4, 1.0),         # Slightly narrow range to ensure enough samples
    'colsample_bytree': (0.4, 1.0),  # Slightly narrow range to ensure enough features
    'reg_lambda': (0, 7),            # Narrow range to reduce over-regularization
    'reg_alpha': (0, 8)              # Narrow range to reduce over-regularization
}

# Initialize Bayesian optimizer
optimizer = BayesianOptimization(
    f=lgb_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=0  # Disable BayesianOptimization's own progress output
)

# Set initial exploration points and number of iterations
init_points = 100
n_iter = 400

# Initial exploration
optimizer.maximize(init_points=init_points, n_iter=0)

# Track best score
best_score = float('-inf')

# Optimize step by step, print score and parameters only when improved
for i in range(n_iter):
    optimizer.maximize(n_iter=1, init_points=0)
    current_score = optimizer.max['target']
    if current_score > best_score:
        best_score = current_score
        print(f"New best score: {best_score:.4f} at iteration {i + init_points + 1}")
        print("Current best parameters:", optimizer.max['params'])

# Get best parameters and convert necessary parameters to int
best_params = optimizer.max['params']
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
best_params['num_leaves'] = int(best_params['num_leaves'])

print("Best parameters:", best_params)

# Rebuild LightGBM model with best parameters and evaluate on test set
best_lgb = LGBMRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
    verbose=-1  # Disable all LightGBM logs
)
best_lgb.fit(X_train, y_train)
y_pred = best_lgb.predict(X_test)
print("Test set R² score:", r2_score(y_test, y_pred))

model_saves_path = os.path.join(model_path, "best_lgb.pkl")
joblib.dump(best_lgb, model_saves_path)
print("Model saved to:", model_path)

### CatBoost

In [ ]:
# Define objective function: calculate mean R² score using 5-fold cross-validation
def catboost_cv(iterations, depth, learning_rate, l2_leaf_reg, bagging_temperature, random_strength):
    # Restrict parameters to reasonable ranges
    iterations = int(iterations)
    depth = int(depth)
    learning_rate = max(min(learning_rate, 1.0), 0.01)  # Limit between 0.01 and 1.0
    l2_leaf_reg = max(l2_leaf_reg, 0)                   # Minimum is 0
    bagging_temperature = max(bagging_temperature, 0)    # Minimum is 0
    random_strength = max(random_strength, 0)           # Minimum is 0
    
    model = CatBoostRegressor(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        bagging_temperature=bagging_temperature,
        random_strength=random_strength,
        random_state=42,
        thread_count=-1,  # Use all available CPU cores
        verbose=0         # Disable training log output
    )
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return cv_score

# Define parameter space
pbounds = {
    'iterations': (400, 800),        # Number of iterations (similar to n_estimators)
    'depth': (3, 10),               # Maximum tree depth
    'learning_rate': (0.05, 0.3),   # Learning rate
    'l2_leaf_reg': (0, 10),         # L2 regularization
    'bagging_temperature': (0, 5),  # Controls bagging randomness
    'random_strength': (0, 5)       # Controls randomness, reduces overfitting
}

# Initialize Bayesian optimizer
optimizer = BayesianOptimization(
    f=catboost_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=0  # Set to 0 to disable BayesianOptimization's own progress output
)

# Set initial exploration points and number of iterations
init_points = 100
n_iter = 400

# Initial exploration
optimizer.maximize(init_points=init_points, n_iter=0)

# Track best score
best_score = float('-inf')

# Optimize step by step, print score and parameters only when improved
for i in range(n_iter):
    optimizer.maximize(n_iter=1, init_points=0)
    current_score = optimizer.max['target']
    if current_score > best_score:
        best_score = current_score
        print(f"New best score: {best_score:.4f} at iteration {i + init_points + 1}")
        print("Current best parameters:", optimizer.max['params'])

# Get best parameters and convert necessary parameters to int
best_params = optimizer.max['params']
best_params['iterations'] = int(best_params['iterations'])
best_params['depth'] = int(best_params['depth'])

print("Best parameters:", best_params)

# Rebuild CatBoost model with best parameters and evaluate on test set
best_cat = CatBoostRegressor(
    **best_params,
    random_state=42,
    thread_count=-1,  # Use all available CPU cores
    verbose=0         # Disable training log output
)

best_cat.fit(X_train, y_train)
y_pred = best_cat.predict(X_test)
print("Test set R² score:", r2_score(y_test, y_pred))

model_saves_path = os.path.join(model_path, "best_cat.pkl")
joblib.dump(best_cat, model_saves_path)
print("Model saved to:", model_path)

### SVR

In [ ]:
# Define objective function: calculate mean R² score using 5-fold cross-validation
def svr_cv(C, epsilon, gamma):
    # Restrict parameters to reasonable ranges
    C = max(C, 0.001)        # Ensure C is not zero or negative
    epsilon = max(epsilon, 0.001)  # Ensure epsilon is not zero or negative
    gamma = max(gamma, 0.001)      # Ensure gamma is not zero or negative
    
    model = SVR(
        C=C,
        epsilon=epsilon,
        gamma=gamma,
        kernel='rbf'  # Use RBF kernel
    )
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return cv_score

# Define parameter space (adjusted based on best results)
pbounds = {
    'C': (5.0, 10.0),         # Focus on larger C values (around 7.298)
    'epsilon': (0.005, 0.05),   # Focus on smaller epsilon values (around 0.01088)
    'gamma': (0.5, 1.0)         # Focus on larger gamma values (around 0.7909)
}

# Initialize Bayesian optimizer
optimizer = BayesianOptimization(
    f=svr_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=0
)

# Set initial exploration points and number of iterations
init_points = 100
n_iter = 400

# Initial exploration
optimizer.maximize(init_points=init_points, n_iter=0)

# Track best score
best_score = float('-inf')

# Optimize step by step, print score and parameters only when improved
for i in range(n_iter):
    optimizer.maximize(n_iter=1, init_points=0)
    current_score = optimizer.max['target']
    if current_score > best_score:
        best_score = current_score
        print(f"New best score: {best_score:.4f} at iteration {i + init_points + 1}")
        print("Current best parameters:", optimizer.max['params'])

# Get best parameters
best_params = optimizer.max['params']
print("Best parameters:", best_params)

# Rebuild SVR model with best parameters and evaluate on test set
best_svr = SVR(
    C=best_params['C'],
    epsilon=best_params['epsilon'],
    gamma=best_params['gamma'],
    kernel='rbf'
)
best_svr.fit(X_train, y_train)
y_pred = best_svr.predict(X_test)
print("Test set R² score:", r2_score(y_test, y_pred))

model_saves_path = os.path.join(model_path, "best_svr.pkl")
joblib.dump(best_svr, model_saves_path)
print("Model saved to:", model_path)

# Result

LightGBM

In [ ]:
load_model_path = os.path.join(model_path,"best_lgb.pkl")
skl()

CatBoost

In [ ]:
load_model_path = os.path.join(model_path,"best_cat.pkl")
skl()

SVR

In [ ]:
load_model_path = os.path.join(model_path,"best_svr.pkl")
skl()